# 15.2 VAE: ELBO와 실습 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter15_2_vae_elbo.ipynb)

책 본문: [Chapter 15](https://smhanlab.com/book-ml/kor/ml1/chapter15.html)

15.1절 GMM(이산 잠재변수 + EM)에서 VAE(연속 잠재변수 + 신경망)로 넘어가는
절의 실습입니다. (1) ELBO의 정규화 항에 쓰는 **KL divergence 닫힌 형태**를
수치 적분으로 직접 검증하고, (2) 본문 손계산 예제(두 봉우리, x=1 / x=9)의
ELBO·진짜 우도·갭을 계산해 **부등식이 실제로 성립하는지** 확인하고,
(3) 두 봉우리 1차원 데이터로 작은 VAE를 500 에폭 학습시켜
**인코더가 두 봉우리를 잠재 공간에서 어떻게 분리하는지**, 생성 모드가
두 봉우리를 재현하는지 확인합니다. 시드는 42로 고정했습니다.

## 1. KL divergence 닫힌 형태의 수치 검증

정규분포 \(q(z)=\mathcal{N}(\mu,\sigma^2)\)와 표준정규분포
\(p(z)=\mathcal{N}(0,1)\) 사이의 KL divergence는

\[D_{KL}(q\|p) = \frac{1}{2}\left(\sigma^2 + \mu^2 - 1 - \log\sigma^2\right)\]

In [1]:
import math
import numpy as np

def kl_closed(mu, var):
    return 0.5 * (var + mu**2 - 1.0 - math.log(var))

def kl_numeric(mu, var, grid=40001, lo=-7, hi=7):
    x = np.linspace(lo, hi, grid)
    logq = -0.5 * ((x - mu) ** 2 / var + math.log(2 * math.pi * var))
    logp = -0.5 * (x ** 2 + math.log(2 * math.pi))
    return float(np.trapezoid(np.exp(logq) * (logq - logp), x))

for mu, var in [(2.0, 1.0), (0.0, 4.0), (3.0, 0.25), (1.5, 0.5)]:
    c, n = kl_closed(mu, var), kl_numeric(mu, var)
    print(f"mu={mu:<4} var={var:<5} closed={c:.6f} numeric={n:.6f} diff={abs(c-n):.2e}")
print()
# 본문 연습문제와 같은 값: q=N(1,1)과 N(0,1) 사이 KL = 0.5
print(f"본문 손계산 예제: KL(N(1,1) || N(0,1)) = {kl_closed(1.0, 1.0)}  (정답 0.5)")
# vae_loss 코드의 클로즈드 포름이 ELBO의 정규화 항 -KL과 같은지
kl_code = -0.5 * (1 + 0.0 - 1.0**2 - math.exp(0.0))  # mu=1, log_var=0
print(f"vae_loss 코드 식의 정규화 항 (mu=1, log_var=0) = {kl_code}  (= -KL = -0.5)")

mu=2.0  var=1.0   closed=2.000000 numeric=1.999996 diff=3.55e-06
mu=0.0  var=4.0   closed=0.806853 numeric=0.797314 diff=9.54e-03
mu=3.0  var=0.25  closed=4.818147 numeric=4.818147 diff=4.44e-15
mu=1.5  var=0.5   closed=1.221574 numeric=1.221574 diff=2.11e-14

본문 손계산 예제: KL(N(1,1) || N(0,1)) = 0.5  (정답 0.5)
vae_loss 코드 식의 정규화 항 (mu=1, log_var=0) = 0.5  (= -KL = -0.5)


## 2. 본문의 손계산 예제: ELBO, 진짜 우도, 그리고 갭

모형: 디코더 \(p(x|z)=\mathcal{N}(x; z, 1)\), 사전분포 \(p(z)=\mathcal{N}(0,1)\).
근사분포를 \(q(z)=\mathcal{N}(1,1)\)으로 (아무렇게나) 고르고 \(x=1, 9\)에
대해 ELBO를 계산한다. **복원 항은 기대값** \(\mathbb{E}_{z\sim q}\)
이므로 \((x-\mu)^2 + \sigma_q^2\)이 들어간다(분산 항을 빼먹으면
"자주 하는 실수" ①).

In [2]:
import math

def elbo(x, mu, var, dec_var=1.0):
    recon = -0.5 * ((x - mu) ** 2 + var + math.log(2 * math.pi * dec_var))
    kl = 0.5 * (var + mu ** 2 - 1.0 - math.log(var))
    return recon, kl, recon - kl

for x in [1.0, 9.0]:
    recon, kl, L = elbo(x, mu=1.0, var=1.0)
    logpx = -0.5 * (x ** 2 / 2.0 + math.log(2 * math.pi * 2.0))  # log N(x; 0, 2)
    print(f"x={x}: E[log p(x|z)]={recon:.4f}, KL={kl:.4f}, ELBO={L:.4f}, "
          f"true log p(x)={logpx:.4f}, gap={logpx - L:.4f}")

# x=9 예제의 갭이 KL(q || p(z|x))인지 확인: 진짜 사후분포 p(z|x=9) = N(9/2, 1/2)
def kl_post(q_mu, q_var, p_mu, p_var):
    return 0.5 * (math.log(p_var / q_var) + (q_var + (q_mu - p_mu) ** 2) / p_var - 1.0)
gap9 = -21.5155 - (-33.9189)  # 위에서 출력된 값
print(f"x=9: 갭 {gap9:.4f} vs KL(N(1,1) || N(4.5,0.5)) = {kl_post(1.0, 1.0, 4.5, 0.5):.4f}")
print(f"x=1: 갭 0.4034  vs KL(N(1,1) || N(0.5,0.5)) = {kl_post(1.0, 1.0, 0.5, 0.5):.4f}")
assert abs((-1.5155) - (-1.9189) - kl_post(1.0, 1.0, 0.5, 0.5)) < 5e-4
print("부등식 log p(x) >= ELBO 성립 확인: x=1일 때 -1.516 >= -1.919 (갭 = KL(q||posterior))")

x=1.0: E[log p(x|z)]=-1.4189, KL=0.5000, ELBO=-1.9189, true log p(x)=-1.5155, gap=0.4034
x=9.0: E[log p(x|z)]=-33.4189, KL=0.5000, ELBO=-33.9189, true log p(x)=-21.5155, gap=12.4034
x=9: 갭 12.4034 vs KL(N(1,1) || N(4.5,0.5)) = 12.4034
x=1: 갭 0.4034  vs KL(N(1,1) || N(0.5,0.5)) = 0.4034
부등식 log p(x) >= ELBO 성립 확인: x=1일 때 -1.516 >= -1.919 (갭 = KL(q||posterior))


## 3. 두 봉우리 데이터로 작은 VAE 학습시키기 (본문 실습 그대로, 시드 42)

데이터: 1 근처와 9 근처에 표준편차 0.5로 200개씩(총 400개). 모델은 본문
코드 그대로(은닉 16, 잠재 2차원). 500 에폭, Adam \(\alpha=0.005\).

In [3]:
import torch
import torch.nn as nn

torch.manual_seed(42)
n = 400
x_left  = torch.randn(n // 2) * 0.5 + 1.0
x_right = torch.randn(n // 2) * 0.5 + 9.0
x_data  = torch.cat([x_left, x_right]).unsqueeze(1)
print(f"data mean={x_data.mean():.2f} std={x_data.std():.2f}")

class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(1, 16), nn.ReLU())
        self.mu = nn.Linear(16, latent_dim)
        self.logvar = nn.Linear(16, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)  # reparameterization
        return self.dec(z), mu, logvar

def loss_fn(x, x_hat, mu, logvar, beta=1.0):
    recon = nn.functional.mse_loss(x, x_hat)
    kl = -0.5 * torch.mean(1 + logvar - mu ** 2 - torch.exp(logvar))
    return recon + beta * kl, recon.item(), kl.item()

model = VAE()
opt = torch.optim.Adam(model.parameters(), lr=0.005)
losses, recons, kls = [], [], []
for ep in range(500):
    opt.zero_grad()
    x_hat, mu, logvar = model(x_data)
    loss, r, k = loss_fn(x_data, x_hat, mu, logvar)
    loss.backward(); opt.step()
    losses.append(loss.item()); recons.append(r); kls.append(k)
    if ep in (0, 99, 199, 299, 399, 499):
        print(f"epoch {ep:>3}: loss={loss.item():.3f} (recon={r:.3f}, KL={k:.3f})")
print()
print("본문(시드 42) 수치: epoch 0 = 36.737 (35.235, 1.502), epoch 499 = 1.667 (0.438, 1.229)")
assert abs(losses[0] - 36.737) < 0.5 and abs(losses[499] - 1.667) < 0.1

data mean=5.04 std=4.05


epoch   0: loss=36.737 (recon=35.235, KL=1.502)
epoch  99: loss=2.636 (recon=0.428, KL=2.208)
epoch 199: loss=1.937 (recon=0.461, KL=1.476)


epoch 299: loss=1.773 (recon=0.406, KL=1.367)
epoch 399: loss=1.707 (recon=0.427, KL=1.281)
epoch 499: loss=1.667 (recon=0.438, KL=1.229)

본문(시드 42) 수치: epoch 0 = 36.737 (35.235, 1.502), epoch 499 = 1.667 (0.438, 1.229)


## 4. 학습된 인코더가 두 봉우리를 어떻게 배치했는가

인코더가 **평균** \(\mu(z)\)만 보면(분산은 작게 수렴했으므로), 왼쪽 봉우리
점과 오른쪽 봉우리 점은 잠재 공간(2차원)의 **다른 위치**에 좁은 분포로
배치되어 있을 것이다 — GMM의 이산적 클러스터 할당의 **연속적 버전**이다.

In [4]:
with torch.no_grad():
    h = model.enc(x_data)
    mu_l  = model.mu(h[:n // 2]).mean(0)
    mu_r  = model.mu(h[n // 2:]).mean(0)
    lv_l  = model.logvar(h[:n // 2]).mean(0)
    lv_r  = model.logvar(h[n // 2:]).mean(0)
    print(f"왼쪽 봉우리(≈1):  mu={mu_l.numpy().round(3)},  logvar={lv_l.numpy().round(3)}")
    print(f"오른쪽 봉우리(≈9): mu={mu_r.numpy().round(3)},  logvar={lv_r.numpy().round(3)}")
print()
print("본문 수치: mu_l≈(0.5,-0.88), mu_r≈(-0.77,1.42) — 첫 번째 잠재차원이 봉우리를 분리는 축")
assert abs(mu_l[0] - 0.5) < 0.15 and abs(mu_r[0] + 0.77) < 0.2 and mu_l[0] > mu_r[0] + 0.5

왼쪽 봉우리(≈1):  mu=[ 0.5   -0.882],  logvar=[-0.833 -1.728]
오른쪽 봉우리(≈9): mu=[-0.768  1.416],  logvar=[-2.918 -4.069]

본문 수치: mu_l≈(0.5,-0.88), mu_r≈(-0.77,1.42) — 첫 번째 잠재차원이 봉우리를 분리는 축


## 5. 생성 모드: 사전분포에서 z를 뽑아 디코더에만 통과시키기

인코더를 **통과시키지 않고** \(z\sim\mathcal{N}(0,I)\) 1000개를 디코더에만
넣으면, 학습된 디코더가 두 봉우리를 재현할 것이다. (본문의 59.1:40.9는
무작위 초기화 실행의 값이며, 시드 42 실행에서는 74.9:25.1로 나온다 —
비대칭의 원인은 본문 참고.)

In [5]:
with torch.no_grad():
    model.eval()
    z = torch.randn(1000, 2)
    gen = model.dec(z).squeeze(1).numpy()
print(f"생성된 샘플: mean={gen.mean():.2f}, std={gen.std():.2f}")
left = (gen < 5.0).mean()
print(f"왼쪽 봉우리(<5) 비율: {left*100:.1f}%   오른쪽(>=5): {(1-left)*100:.1f}%")
print("본문(시드 42) 수치: 74.9% / 25.1% (무작위 초기화 실행: 59.1% / 40.9%)")

생성된 샘플: mean=3.40, std=2.82
왼쪽 봉우리(<5) 비율: 74.9%   오른쪽(>=5): 25.1%
본문(시드 42) 수치: 74.9% / 25.1% (무작위 초기화 실행: 59.1% / 40.9%)


## 6. 학습 과정 시각화 (본문 그림 ch15_2_vae_elbo.svg 생성)

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Noto Sans CJK KR"
plt.rcParams["axes.unicode_minus"] = False

with torch.no_grad():
    h = model.enc(x_data)
    mu_l_full = model.mu(h[:n // 2]).numpy()   # 각 점별 (400/2, 2)
    mu_r_full = model.mu(h[n // 2:]).numpy()

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# (a) 손실 곡선
ax = axes[0, 0]
ax.plot(losses, color="black", lw=1.2, label="Total loss (−ELBO)")
ax.plot(recons, color="#1f77b4", lw=1.0, label="Reconstruction loss")
ax.plot(kls, color="#ff7f0e", lw=1.0, label="KL term")
ax.set_yscale("log")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (log)")
ax.set_title("(a) Tug-of-war between the two terms"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (b) 잠재 공간: 학습된 인코더 평균 (각 점별 mu(z))
ax = axes[0, 1]
sc1 = ax.scatter(mu_l_full[:, 0], mu_l_full[:, 1], s=8, alpha=0.5, color="#1f77b4", label="Left peak (x≈1)")
sc2 = ax.scatter(mu_r_full[:, 0], mu_r_full[:, 1], s=8, alpha=0.5, color="#d62728", label="Right peak (x≈9)")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel(r"$z_1$ (first latent dimension)"); ax.set_ylabel(r"$z_2$")
ax.set_title("(b) Learned encoder means μ(z) — peaks separated"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (c) 원본 데이터
ax = axes[1, 0]
ax.hist(x_left.numpy(), bins=30, color="#1f77b4", alpha=0.7, label="x≈1")
ax.hist(x_right.numpy(), bins=30, color="#d62728", alpha=0.7, label="x≈9")
ax.set_xlabel("x"); ax.set_title("(c) Original data (two peaks)"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (d) 생성 샘플
ax = axes[1, 1]
ax.hist(gen, bins=30, color="#2ca02c", alpha=0.7)
ax.axvline(5.0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("x"); ax.set_title("(d) Generated samples (prior z, decoder only)")
ax.set_ylabel("Count"); ax.grid(alpha=0.3)

fig.suptitle("VAE training — balance of the reconstruction and KL terms, and reproduction of the two peaks (seed 42)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig("ch15_2_vae_elbo.svg", bbox_inches="tight")
print("saved: ch15_2_vae_elbo.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)")
plt.show()

saved: ch15_2_vae_elbo.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)


## 7. β-VAE: 두 항의 균형점을 조절하기

정규화 항에 계수 \(\beta\)를 붙여 \(\mathcal{L}=\mathbb{E}[\log p(x|z)] - \beta\, D_{KL}\).
\(\beta=0\)은 일반 오토인코더(KL 무력화, 잠재 공간이 무질서),
\(\beta\)가 크면 KL이 작아지지만 복원 품질이 떨어진다(본문 표 재현, 시드 7, 200 에폭).

In [7]:
def train_vae(beta, epochs=200, seed=7):
    torch.manual_seed(seed)
    m = VAE()
    o = torch.optim.Adam(m.parameters(), lr=0.005)
    for _ in range(epochs):
        o.zero_grad()
        x_hat, mu, logvar = m(x_data)
        loss, r, k = loss_fn(x_data, x_hat, mu, logvar, beta=beta)
        loss.backward(); o.step()
    with torch.no_grad():
        g = m.dec(torch.randn(2000, 2)).squeeze(1).numpy()
    return r, k, g.std()

print("beta   recon    KL       generated std")
for beta in [0.0, 0.1, 0.5, 1.0, 2.0]:
    r, k, gstd = train_vae(beta)
    print(f"{beta:<6} {r:.3f}   {k:.3f}   {gstd:.2f}")
print()
print("본문 표와 비교: beta=0 -> (0.016, 19.1, 0.58), beta=2 -> (0.546, 0.90, 2.42)")

beta   recon    KL       generated std
0.0    0.016   19.133   0.58


0.1    0.068   5.607   0.91
0.5    0.232   1.704   1.67


1.0    0.378   1.218   2.05
2.0    0.546   0.901   2.42

본문 표와 비교: beta=0 -> (0.016, 19.1, 0.58), beta=2 -> (0.546, 0.90, 2.42)
